## 👥 CrewAI Lab
Build collaborative multi-agent systems using CrewAI for support and task-based workflows.

In [2]:
import sqlite3
from crewai import Agent
from crewai_tools import BaseTool
from langchain_community.llms import Ollama


class OrderQueryTool(BaseTool):
    name:str = "OrderQuery"
    description: str = "Queries order details by customer name."
    def _run(self, customer:str) ->str:
        conn = sqlite3.connect("data/orders.db")
        cursor = conn.cursor()
        cursor().execute("SELECT id, item, price, status FROM orders WHERE customer = ?", (customer,))
        result = cursor.fetchone()
        conn.close()
        return f"Order: ID {result[0]}, Item: {result[1]}, Price: ${result[2]}, Status: {result[3]}" if result else "No order found."

llm = Ollama(model= "mistral")

support_agent = Agent(
    role="Customer Support Specialist",
    goal="Answer customer queries about orders accurately.",
    backstory="Expert in e-commerce support with deep product knowledge.",
    tools=[OrderQueryTool()],
    llm=llm,
    verbose=True
)

# Test agent
result = support_agent.execute_task("What is Alice's order status?")
print("Query: What is Alice's order status?\nAnswer:", result)











    

/opt/anaconda3/envs/whisper_env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:898: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `Adapter` to V2.
  warn(


ValidationError: 1 validation error for Agent
tools.0
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=OrderQueryTool(name='Orde...ame.', args_schema=None), input_type=OrderQueryTool]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type

In [12]:
from crewai import Agent, Task , Crew
from langchain_community.llms import Ollama

# Initialize LLM
llm = Ollama(model="mistral")

# Define agent
refund_agent = Agent(
    role="Refund Specialist",
    goal="Determine refund eligibility based on policy.",
    backstory="Expert in refund policies and customer satisfaction.",
    llm=llm,
    verbose=True
)

# Define task
refund_task = Task(
    description="Check if Alice's order is eligible for a refund based on the policy: Refunds within 30 days for delivered items.",
    expected_output="A clear statement on refund eligibility.",
    agent=refund_agent
)

crew = Crew(
    agents=[refund_agent],
    tasks=[refund_task],
    verbose=True
)

result = crew.kickoff()
print("Task result:", result)



╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 52a2a181-e81f-4f34-8e00-8adac8e9a11d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: dcb1928b-4630-4d7f-9e36-e681376f0c64
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: dcb1928b-4630-4d7f-9e36-e681376f0c64
       Status: Executing Task...
    └── 🤖 Agent: Refund Specialist
            Status: In Progress

# Agent: Refund Specialist
## Task: Check if Alice's order is eligible for a refund based on the policy: Refunds within 30 days for delivered items.


🤖 Agent: Refund Specialist
    Status: In Progress
└── 🧠 Thinking...

...

 Error during LLM call: litellm.BadRequestError: GetLLMProvider Exception - list index out of range

original model: mistral
 An unknown error occurred. Please check the details below.
 Error details: litellm.BadRequestError: GetLLMProvider Exception - list index out of range

original model: mistral


🚀 Crew: crew
└── 📋 Task: dcb1928b-4630-4d7f-9e36-e681376f0c64
       Assigned to: Refund Specialist
       Status: ❌ Failed
    └── 🤖 Agent: Refund Specialist
            Status: In Progress
        └── ❌ LLM Failed

╭───────────────────────────────────────────────── Task Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: dcb1928b-4630-4d7f-9e36-e681376f0c64                                                                     │
│  Agent: Refund Specialist                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 52a2a181-e81f-4f34-8e00-8adac8e9a11d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

BadRequestError: litellm.BadRequestError: GetLLMProvider Exception - list index out of range

original model: mistral

In [14]:
from crewai import Agent, Task, Crew
from crewai_tools import BaseTool
from langchain_community.llms import Ollama

# Simulated web search tool
class WebSearchTool(BaseTool):
    name: str = "WebSearch"
    description: str = "Simulates searching the web for product reviews."

    def _run(self, query: str) -> str:
        return "Web reviews: Customers love the Laptop's display and performance."

# Initialize LLM
llm = Ollama(model="mistral")

# Define agent
review_agent = Agent(
    role="Review Analyst",
    goal="Gather product review insights.",
    backstory="Expert in analyzing customer feedback.",
    tools=[WebSearchTool()],
    llm=llm,
    verbose=True
)

# Define task
review_task = Task(
    description="Find reviews for the Laptop product.",
    expected_output="A summary of customer reviews.",
    agent=review_agent
)

# Create crew
crew = Crew(
    agents=[review_agent],
    tasks=[review_task],
    verbose=True
)

# Execute crew
result = crew.kickoff()
print("Task: Find Laptop reviews\nAnswer:", result)

ValidationError: 1 validation error for Agent
tools.0
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=WebSearchTool(name='WebSe...ews.', args_schema=None), input_type=WebSearchTool]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type

In [16]:
import sqlite3
from crewai import Agent, Task, Crew
from crewai_tools import BaseTool
from langchain_community.llms import Ollama

# Database tool
class OrderQueryTool(BaseTool):
    name: str = "OrderQuery"
    description: str = "Queries order details by customer name."

    def _run(self, customer: str) -> str:
        conn = sqlite3.connect("data/orders.db")
        cursor = conn.cursor()
        cursor.execute("SELECT id, item, price, status FROM orders WHERE customer = ?", (customer,))
        result = cursor.fetchone()
        conn.close()
        return f"Order: ID {result[0]}, Item: {result[1]}, Price: ${result[2]}, Status: {result[3]}" if result else "No order found."

# Initialize LLM
llm = Ollama(model="mistral")

# Define agent with memory
support_agent = Agent(
    role="Customer Support Specialist",
    goal="Answer queries with context from past interactions.",
    backstory="Expert in personalized support.",
    tools=[OrderQueryTool()],
    llm=llm,
    memory=True,
    verbose=True
)

# Define tasks
task1 = Task(
    description="Check the status of Alice's order.",
    expected_output="A summary of Alice's order status.",
    agent=support_agent
)

task2 = Task(
    description="Based on Alice's order, can she get a discount?",
    expected_output="A statement on discount eligibility.",
    agent=support_agent
)

# Create crew
crew = Crew(
    agents=[support_agent],
    tasks=[task1, task2],
    verbose=True
)

# Execute crew
result = crew.kickoff()
print("Crew Output:", result)

ValidationError: 1 validation error for Agent
tools.0
  Input should be a valid dictionary or instance of BaseTool [type=model_type, input_value=OrderQueryTool(name='Orde...ame.', args_schema=None), input_type=OrderQueryTool]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type

In [2]:
from crewai import Agent, Task, Crew
from crewai.knowledge.source.file_knowledge_source import FileKnowledgeSource
from langchain_community.llms import Ollama

# Knowledge source
product_info = FileKnowledgeSource(
    file_path="data/product_info.txt",
    metadata={"category": "product_specs"}
)

# Initialize LLM
llm = Ollama(model="mistral")

# Define agent
support_agent = Agent(
    role="Customer Support Specialist",
    goal="Answer product queries with detailed specs.",
    backstory="Expert in product details.",
    knowledge_sources=[product_info],
    llm=llm,
    verbose=True
)

# Define task
task = Task(
    description="What are the specs of the Laptop?",
    expected_output="A detailed description of the Laptop's specs.",
    agent=support_agent
)

# Create crew
crew = Crew(
    agents=[support_agent],
    tasks=[task],
    verbose=True
)

# Execute crew
result = crew.kickoff()
print("Task: Laptop specs\nAnswer:", result)

ModuleNotFoundError: No module named 'crewai.knowledge.source.file_knowledge_source'

In [4]:
import sqlite3
from crewai import Agent, Task, Crew, Process
from crewai_tools import BaseTool
from crewai.knowledge.source.file_knowledge_source import FileKnowledgeSource
from langchain_community.llms import Ollama
from transformers import pipeline

# Tools
class OrderQueryTool(BaseTool):
    name: str = "OrderQuery"
    description: str = "Queries order details by customer name."

    def _run(self, customer: str) -> str:
        conn = sqlite3.connect("data/orders.db")
        cursor = conn.cursor()
        cursor.execute("SELECT id, item, price, status FROM orders WHERE customer = ?", (customer,))
        result = cursor.fetchone()
        conn.close()
        return f"Order: ID {result[0]}, Item: {result[1]}, Price: ${result[2]}, Status: {result[3]}" if result else "No order found."

class SentimentTool(BaseTool):
    name: str = "SentimentAnalysis"
    description: str = "Analyzes sentiment of customer queries."

    def _run(self, text: str) -> str:
        analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
        result = analyzer(text)[0]
        return f"Sentiment: {result['label']} (Confidence: {result['score']:.2f})"

# Knowledge source
policy_info = FileKnowledgeSource(file_path="data/support_policy.txt", metadata={"category": "policy"})

# Initialize LLM
llm = Ollama(model="mistral")

# Define agents
support_agent = Agent(
    role="Customer Support Specialist",
    goal="Answer queries about orders and products.",
    backstory="Expert in e-commerce support.",
    tools=[OrderQueryTool()],
    llm=llm,
    memory=True,
    verbose=True
)

refund_agent = Agent(
    role="Refund Specialist",
    goal="Handle refund requests.",
    backstory="Expert in refund policies.",
    knowledge_sources=[policy_info],
    llm=llm,
    memory=True,
    verbose=True
)

sentiment_agent = Agent(
    role="Sentiment Analyst",
    goal="Analyze customer query sentiment.",
    backstory="Expert in customer emotions.",
    tools=[SentimentTool()],
    llm=llm,
    verbose=True
)

# Define tasks
sentiment_task = Task(
    description="Analyze the sentiment of this query: 'I love my new laptop but want a refund.'",
    expected_output="A sentiment analysis result.",
    agent=sentiment_agent
)

status_task = Task(
    description="Check the status of Alice's order.",
    expected_output="A summary of Alice's order status.",
    agent=support_agent
)

refund_task = Task(
    description="Check if Alice's order is eligible for a refund.",
    expected_output="A statement on refund eligibility.",
    agent=refund_agent
)

# Create crew
crew = Crew(
    agents=[sentiment_agent, support_agent, refund_agent],
    tasks=[sentiment_task, status_task, refund_task],
    process=Process.sequential,
    verbose=True,
    planning=True
)

# Execute crew
result = crew.kickoff()
print("Crew Output:", result)

/opt/anaconda3/envs/whisper_env/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:898: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `Adapter` to V2.
  warn(


ModuleNotFoundError: No module named 'crewai.knowledge.source.file_knowledge_source'